# Support functions for ROI + 2.5D lumbar classification

Reusable imports, helpers, dataset, model factory, training utilities, and evaluation helpers used by the main notebook.

## Imports

In [ ]:
# IMPORTS
import os
import re
import copy
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import cv2
import pydicom
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from torchvision import transforms, models

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report,
    log_loss,
    roc_curve,
    auc,
    roc_auc_score,
)

from tqdm.auto import tqdm
from sklearn.preprocessing import label_binarize


## Reproducibility

In [ ]:
# SET SEED FOR REPRODUCIBILITY
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(42)

## Dataframe helpers

In [ ]:
# DATAFRAME AND METADATA HELPERS

def extract_side(condition):
    condition = str(condition).lower()

    if condition.startswith("left"):
        return "left"
    elif condition.startswith("right"):
        return "right"
    else:
        return "center"


def extract_base_condition(condition):
    condition = str(condition)
    return (
        condition
        .replace("Left ", "")
        .replace("Right ", "")
    )


def infer_coordinate_columns(df):
    """Infer coordinate columns used for ROI cropping."""
    x_candidates = ["x", "coord_x", "x_coord", "coordinate_x", "keypoint_x", "image_x"]
    y_candidates = ["y", "coord_y", "y_coord", "coordinate_y", "keypoint_y", "image_y"]

    x_col = next((col for col in x_candidates if col in df.columns), None)
    y_col = next((col for col in y_candidates if col in df.columns), None)

    return x_col, y_col


def infer_series_group_columns(df):
    """
    Infer columns that identify one DICOM series.

    Best case: study_id + series_id / series_instance_uid.
    Fallback: study_id + series_description. The fallback can mix repeated series
    of the same type within one study, so the detected columns are printed below.
    """
    series_candidates = [
        "series_id",
        "SeriesInstanceUID",
        "series_instance_uid",
        "series_uid",
    ]

    for series_col in series_candidates:
        if series_col in df.columns:
            return ["study_id", series_col]

    return ["study_id", "series_description"]


def infer_slice_order_column(df):
    """Infer column used to order slices within a series."""
    candidates = [
        "instance_number",
        "InstanceNumber",
        "instance",
        "slice_number",
        "slice_idx",
        "image_number",
        "image_index",
        "dicom_instance_number",
        "z",
    ]

    return next((col for col in candidates if col in df.columns), None)


def parse_slice_order_from_path(path):
    """
    Fallback ordering when no DICOM instance column exists.
    Extracts the final integer from the file stem/path.
    """
    matches = re.findall(r"\d+", str(path))
    if len(matches) == 0:
        return np.nan
    return int(matches[-1])


def make_model_df(df, base_condition, series_description):
    """Filter the master dataframe to the condition/series used by one classifier."""
    out = df[
        (df["base_condition"] == base_condition) &
        (df["series_description"] == series_description)
    ].copy()

    out = out.dropna(subset=[
        "img_path",
        "target",
        "level",
        "side",
        "series_description",
    ])

    return out.reset_index(drop=True)


## DICOM, ROI crop, and 2.5D dataset

In [ ]:
def read_dicom_array(path, clip_percentiles=(1, 99)):
    """Read one DICOM image and return a normalized float32 grayscale array in [0, 1]."""
    ds = pydicom.dcmread(path)

    img = ds.pixel_array.astype(np.float32)

    # Correct display inversion for MONOCHROME1.
    if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
        img = img.max() - img

    # Robust clipping reduces the influence of extreme values.
    low, high = np.percentile(img, clip_percentiles)
    if high > low:
        img = np.clip(img, low, high)

    # Normalize to [0, 1].
    img = img - img.min()
    denom = img.max()
    if denom > 0:
        img = img / denom

    return img.astype(np.float32)


def crop_array_around_xy(img, x, y, crop_size=ROI_CROP_SIZE, fallback_to_full_image=ROI_FALLBACK_TO_FULL_IMAGE):
    """
    Crop a square ROI around (x, y) from a 2D array.

    Coordinates are expected in the original DICOM pixel space.
    If coordinates are missing and fallback_to_full_image=True, the full image is returned.
    Crops going outside image boundaries are padded with zeros.
    """
    if x is None or y is None or pd.isna(x) or pd.isna(y):
        if fallback_to_full_image:
            return img
        raise ValueError("Missing ROI coordinates and fallback_to_full_image=False.")

    h, w = img.shape[:2]
    x = float(x)
    y = float(y)

    half = int(round(crop_size / 2))
    cx = int(round(x))
    cy = int(round(y))

    x1 = cx - half
    x2 = x1 + crop_size
    y1 = cy - half
    y2 = y1 + crop_size

    src_x1 = max(x1, 0)
    src_x2 = min(x2, w)
    src_y1 = max(y1, 0)
    src_y2 = min(y2, h)

    crop = np.zeros((crop_size, crop_size), dtype=img.dtype)

    dst_x1 = src_x1 - x1
    dst_x2 = dst_x1 + (src_x2 - src_x1)
    dst_y1 = src_y1 - y1
    dst_y2 = dst_y1 + (src_y2 - src_y1)

    if src_x2 > src_x1 and src_y2 > src_y1:
        crop[dst_y1:dst_y2, dst_x1:dst_x2] = img[src_y1:src_y2, src_x1:src_x2]
    elif fallback_to_full_image:
        return img

    return crop


def array_to_uint8(img):
    return (img * 255).clip(0, 255).astype(np.uint8)


def resize_array_to_shape(img, target_shape):
    """Resize a 2D array to target_shape=(height, width) if needed."""
    target_h, target_w = target_shape

    if img.shape[:2] == (target_h, target_w):
        return img

    return cv2.resize(
        img,
        (target_w, target_h),
        interpolation=cv2.INTER_LINEAR,
    ).astype(np.float32)


def stack_grayscale_arrays_as_rgb(arrays):
    """
    Stack three grayscale arrays into a 3-channel PIL image.

    For 2.5D, channels are usually previous/current/next slices.
    Shapes are aligned to the middle/current channel before stacking.
    """
    if len(arrays) != 3:
        raise ValueError(f"Expected exactly 3 arrays, got {len(arrays)}.")

    target_shape = arrays[1].shape[:2]
    arrays = [resize_array_to_shape(arr, target_shape) for arr in arrays]

    stacked = np.stack(arrays, axis=-1)
    stacked_uint8 = array_to_uint8(stacked)

    return Image.fromarray(stacked_uint8, mode="RGB")


def read_dicom_as_pil_grayscale(path):
    """Read DICOM and convert to PIL grayscale image."""
    img = read_dicom_array(path)
    img_uint8 = array_to_uint8(img)
    return Image.fromarray(img_uint8).convert("L")


def read_dicom_as_pil_roi_crop(row):
    """Read DICOM, crop around row coordinates, and convert to PIL grayscale image."""
    img = read_dicom_array(row["img_path"])

    if not ROI_CROP_ENABLED or X_COLUMN is None or Y_COLUMN is None:
        crop = img
    else:
        crop = crop_array_around_xy(
            img,
            row.get(X_COLUMN, np.nan),
            row.get(Y_COLUMN, np.nan),
            crop_size=ROI_CROP_SIZE,
            fallback_to_full_image=ROI_FALLBACK_TO_FULL_IMAGE,
        )

    crop_uint8 = array_to_uint8(crop)
    return Image.fromarray(crop_uint8).convert("L")


def load_single_slice_array_for_model(row, img_path=None, use_roi_crop=ROI_CROP_ENABLED):
    """
    Load one DICOM slice as a 2D array after optional ROI cropping.

    For neighbouring 2.5D slices, the neighbour image path changes but
    the ROI coordinate is taken from the current labelled row.
    """
    path = img_path if img_path is not None else row["img_path"]
    img = read_dicom_array(path)

    if use_roi_crop and X_COLUMN is not None and Y_COLUMN is not None:
        img = crop_array_around_xy(
            img,
            row.get(X_COLUMN, np.nan),
            row.get(Y_COLUMN, np.nan),
            crop_size=ROI_CROP_SIZE,
            fallback_to_full_image=ROI_FALLBACK_TO_FULL_IMAGE,
        )

    return img


def get_roi_box_for_display(row):
    """Return a clipped ROI rectangle for visual sanity checks."""
    img = read_dicom_array(row["img_path"])
    h, w = img.shape[:2]

    if not ROI_CROP_ENABLED or X_COLUMN is None or Y_COLUMN is None:
        return img, None

    x = row.get(X_COLUMN, np.nan)
    y = row.get(Y_COLUMN, np.nan)

    if pd.isna(x) or pd.isna(y):
        return img, None

    half = int(round(ROI_CROP_SIZE / 2))
    cx = int(round(float(x)))
    cy = int(round(float(y)))

    x1 = max(cx - half, 0)
    y1 = max(cy - half, 0)
    x2 = min(cx - half + ROI_CROP_SIZE, w)
    y2 = min(cy - half + ROI_CROP_SIZE, h)

    return img, (x1, y1, x2, y2)


In [ ]:
class LumbarMetadataClassificationDataset(Dataset):
    def __init__(
        self,
        df,
        transform=None,
        augmentation_transform=None,
        augment=False,
        use_roi_crop=ROI_CROP_ENABLED,
        use_2_5d_input=USE_2_5D_INPUT,
        slice_context_offsets=SLICE_CONTEXT_OFFSETS,
        series_group_columns=None,
        slice_order_column=None,
        level_to_id=None,
        side_to_id=None,
        series_to_id=None,
    ):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.augmentation_transform = augmentation_transform
        self.augment = augment
        self.use_roi_crop = use_roi_crop
        self.use_2_5d_input = use_2_5d_input
        self.slice_context_offsets = tuple(slice_context_offsets)

        self.series_group_columns = (
            series_group_columns
            if series_group_columns is not None
            else SERIES_GROUP_COLUMNS
        )
        self.slice_order_column = (
            slice_order_column
            if slice_order_column is not None
            else SLICE_ORDER_COLUMN
        )

        self.level_to_id = level_to_id if level_to_id is not None else LEVEL_TO_ID
        self.side_to_id = side_to_id if side_to_id is not None else SIDE_TO_ID
        self.series_to_id = series_to_id if series_to_id is not None else SERIES_TO_ID

        self.series_groups = {}
        self.path_position_lookup = {}

        if self.use_2_5d_input:
            self._build_series_index()

    def _as_group_key(self, values):
        if isinstance(values, tuple):
            return values
        return (values,)

    def _row_group_key(self, row):
        return tuple(row[col] for col in self.series_group_columns)

    def _build_series_index(self):
        """
        Build per-series ordered unique-slice tables for neighbouring-slice lookup.

        The labelled dataframe can contain multiple rows per image because several
        levels/sides/conditions can point to the same DICOM slice. For slice context,
        we need a unique ordered list of image paths per series.
        """
        if self.series_group_columns is None or self.slice_order_column is None:
            print("WARNING: 2.5D indexing unavailable. Falling back to repeated current slice.")
            return

        required_cols = list(self.series_group_columns) + [self.slice_order_column, "img_path"]
        missing_cols = [col for col in required_cols if col not in self.df.columns]

        if len(missing_cols) > 0:
            print(f"WARNING: Missing columns for 2.5D indexing: {missing_cols}. Falling back to repeated current slice.")
            return

        for key, group in self.df.groupby(self.series_group_columns, dropna=False):
            key = self._as_group_key(key)

            group = (
                group[required_cols]
                .dropna(subset=["img_path"])
                .drop_duplicates(subset=["img_path"])
                .copy()
            )

            group[self.slice_order_column] = pd.to_numeric(
                group[self.slice_order_column],
                errors="coerce",
            )

            # If the order is missing for a few files, use filename-derived order.
            if group[self.slice_order_column].isna().any():
                missing_mask = group[self.slice_order_column].isna()
                group.loc[missing_mask, self.slice_order_column] = group.loc[
                    missing_mask,
                    "img_path",
                ].apply(parse_slice_order_from_path)

            group = group.sort_values([self.slice_order_column, "img_path"]).reset_index(drop=True)

            self.series_groups[key] = group

            for pos, row in group.iterrows():
                self.path_position_lookup[(key, row["img_path"])] = pos

    def __len__(self):
        return len(self.df)

    def _get_neighbor_paths(self, row):
        """
        Return paths for previous/current/next context.

        If series lookup fails, repeat the current image path so the sample is still valid.
        """
        current_path = row["img_path"]

        if not self.use_2_5d_input or len(self.series_groups) == 0:
            return [current_path, current_path, current_path]

        try:
            key = self._row_group_key(row)
            group = self.series_groups[key]
            pos = self.path_position_lookup[(key, current_path)]
        except Exception:
            return [current_path, current_path, current_path]

        paths = []
        for offset in self.slice_context_offsets:
            neighbor_pos = int(np.clip(pos + offset, 0, len(group) - 1))
            paths.append(group.iloc[neighbor_pos]["img_path"])

        return paths

    def _load_model_image(self, row):
        """
        Load model image as RGB PIL.

        - 2.5D enabled: RGB channels = previous/current/next slices.
        - 2.5D disabled: RGB channels = current grayscale slice repeated.
        """
        if self.use_2_5d_input:
            paths = self._get_neighbor_paths(row)
            arrays = [
                load_single_slice_array_for_model(
                    row,
                    img_path=path,
                    use_roi_crop=self.use_roi_crop,
                )
                for path in paths
            ]
            image = stack_grayscale_arrays_as_rgb(arrays)
        else:
            if self.use_roi_crop:
                image = read_dicom_as_pil_roi_crop(row)
            else:
                image = read_dicom_as_pil_grayscale(row["img_path"])

            # Make sure the pretrained 3-channel backbone always receives 3 channels.
            image = image.convert("RGB")

        return image

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = self._load_model_image(row)

        if self.augment and self.augmentation_transform is not None:
            image = self.augmentation_transform(image)

        if self.transform is not None:
            image = self.transform(image)

        metadata = {
            "level": torch.tensor(self.level_to_id[row["level"]], dtype=torch.long),
            "side": torch.tensor(self.side_to_id[row["side"]], dtype=torch.long),
            "series": torch.tensor(self.series_to_id[row["series_description"]], dtype=torch.long),
        }

        target = torch.tensor(int(row["target"]), dtype=torch.long)

        return image, metadata, target


## Mean/std computation

In [ ]:
# Transform used only for mean/std calculation.
# No normalization here.
# Important: no Grayscale() here, because with 2.5D input the 3 channels are
# neighbouring slices and must not be collapsed back into repeated grayscale.
mean_std_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])


@torch.no_grad()
def compute_mean_std(df, batch_size=32, num_workers=2):
    dataset = LumbarMetadataClassificationDataset(
        df,
        transform=mean_std_transform,
        use_roi_crop=ROI_CROP_ENABLED,
        use_2_5d_input=USE_2_5D_INPUT,
    )

    loader_kwargs = {
        "dataset": dataset,
        "batch_size": batch_size,
        "shuffle": False,
        "num_workers": num_workers,
        "pin_memory": PIN_MEMORY,
        "persistent_workers": (num_workers > 0),
    }
    if num_workers > 0:
        loader_kwargs["prefetch_factor"] = PREFETCH_FACTOR

    loader = DataLoader(**loader_kwargs)

    channel_sum = torch.zeros(3)
    channel_sum_sq = torch.zeros(3)
    num_pixels = 0

    for images, metadata, targets in tqdm(loader, desc="Computing mean/std"):
        # images shape: [B, C, H, W]
        images = images.float()
        b, c, h, w = images.shape

        channel_sum += images.sum(dim=(0, 2, 3))
        channel_sum_sq += (images ** 2).sum(dim=(0, 2, 3))
        num_pixels += b * h * w

    mean = channel_sum / num_pixels
    variance = channel_sum_sq / num_pixels - mean ** 2
    std = torch.sqrt(torch.clamp(variance, min=1e-8))

    return mean.tolist(), std.tolist()


## Visual sanity-check helpers

In [ ]:
# ============================================================
# VISUAL SANITY CHECK — ACTUAL DATASET OUTPUT
# ============================================================
#
# This visualizes the same dataset pipeline used for training:
# DICOM -> optional ROI crop -> optional 2.5D neighbouring-slice channels
# -> resize -> tensor -> normalization.
#
# The displayed tensor is denormalized only for visualization.

def get_normalize_from_transform(transform):
    if isinstance(transform, transforms.Compose):
        for t in transform.transforms:
            if isinstance(t, transforms.Normalize):
                mean_t = torch.tensor(t.mean).view(3, 1, 1)
                std_t = torch.tensor(t.std).view(3, 1, 1)
                return mean_t, std_t

    return torch.zeros(3, 1, 1), torch.ones(3, 1, 1)


def tensor_to_display_image(image_tensor, transform):
    image_tensor = image_tensor.detach().cpu()

    mean_t, std_t = get_normalize_from_transform(transform)
    image_tensor = image_tensor * std_t + mean_t
    image_tensor = torch.clamp(image_tensor, 0, 1)

    return image_tensor.permute(1, 2, 0).numpy()


def plot_dataset_input_samples(
    dataset,
    n_samples=6,
    random_state=42,
):
    n_samples = min(n_samples, len(dataset))

    rng = np.random.default_rng(random_state)
    sample_indices = rng.choice(len(dataset), size=n_samples, replace=False)

    fig, axes = plt.subplots(
        n_samples,
        4,
        figsize=(14, 3.5 * n_samples),
    )

    if n_samples == 1:
        axes = np.expand_dims(axes, axis=0)

    for row_idx, idx in enumerate(sample_indices):
        row = dataset.df.iloc[idx]

        # 1) Original image with ROI box.
        original_img, box = get_roi_box_for_display(row)

        ax = axes[row_idx, 0]
        ax.imshow(original_img, cmap="gray")
        ax.axis("off")
        ax.set_title(
            f"Original + ROI\n"
            f"{row['base_condition']} | {row['level']} | {row['side']} | {row['severity']}",
            fontsize=8,
        )

        if box is not None:
            x1, y1, x2, y2 = box
            ax.add_patch(
                plt.Rectangle(
                    (x1, y1),
                    x2 - x1,
                    y2 - y1,
                    fill=False,
                    linewidth=2,
                    edgecolor="red",
                )
            )

            if (
                X_COLUMN is not None
                and Y_COLUMN is not None
                and pd.notna(row[X_COLUMN])
                and pd.notna(row[Y_COLUMN])
            ):
                ax.scatter([row[X_COLUMN]], [row[Y_COLUMN]], s=25, c="yellow")

        # 2) Raw model PIL image before resize/tensor/normalization.
        #    For 2.5D, this is an RGB image where channels are neighbouring slices.
        raw_model_pil = dataset._load_model_image(row)
        raw_model_np = np.asarray(raw_model_pil)

        ax = axes[row_idx, 1]
        ax.imshow(raw_model_np)
        ax.axis("off")
        ax.set_title(
            f"Raw model image\nshape={raw_model_np.shape}",
            fontsize=8,
        )

        # 3) Show the three input channels separately.
        image_tensor, metadata, label = dataset[idx]
        display_img = tensor_to_display_image(image_tensor, dataset.transform)

        channel_titles = (
            ["prev", "current", "next"]
            if USE_2_5D_INPUT
            else ["gray", "gray", "gray"]
        )

        channel_panel = np.concatenate(
            [display_img[:, :, c] for c in range(3)],
            axis=1,
        )

        ax = axes[row_idx, 2]
        ax.imshow(channel_panel, cmap="gray")
        ax.axis("off")
        ax.set_title(
            f"Final channels: {channel_titles}\n"
            f"tensor={tuple(image_tensor.shape)}",
            fontsize=8,
        )

        # 4) Actual model RGB tensor after denormalization for display.
        ax = axes[row_idx, 3]
        ax.imshow(display_img)
        ax.axis("off")
        ax.set_title(
            f"Actual model input\nlabel={int(label)}",
            fontsize=8,
        )

    plt.tight_layout()
    plt.show()


## Model factory and optimizer

In [ ]:
# IMAGE + METADATA MODEL FACTORY
#
# The selected image backbone is controlled in the main notebook by SELECTED_BACKBONE.
# All supported models keep the same metadata fusion head, so the training workflow
# remains unchanged when switching architecture.
#
# Supported backbone families:
#   - ResNet
#   - DenseNet
#   - EfficientNet
#   - Swin Transformer
#   - ConvNeXt
#
# The dictionary stores torchvision function names and weight enum names as strings.
# This keeps the cell robust across torchvision versions: unavailable models fail
# only when selected, with a clear error message.

SUPPORTED_BACKBONES = {
    # ----------------------------
    # ResNet family
    # ----------------------------
    "resnet18": {
        "builder_name": "resnet18",
        "weights_enum_name": "ResNet18_Weights",
        "family": "resnet",
    },
    "resnet34": {
        "builder_name": "resnet34",
        "weights_enum_name": "ResNet34_Weights",
        "family": "resnet",
    },
    "resnet50": {
        "builder_name": "resnet50",
        "weights_enum_name": "ResNet50_Weights",
        "family": "resnet",
    },

    # ----------------------------
    # DenseNet family
    # ----------------------------
    "densenet121": {
        "builder_name": "densenet121",
        "weights_enum_name": "DenseNet121_Weights",
        "family": "densenet",
    },
    "densenet169": {
        "builder_name": "densenet169",
        "weights_enum_name": "DenseNet169_Weights",
        "family": "densenet",
    },
    "densenet201": {
        "builder_name": "densenet201",
        "weights_enum_name": "DenseNet201_Weights",
        "family": "densenet",
    },

    # ----------------------------
    # EfficientNet family
    # ----------------------------
    "efficientnet_b0": {
        "builder_name": "efficientnet_b0",
        "weights_enum_name": "EfficientNet_B0_Weights",
        "family": "efficientnet",
    },
    "efficientnet_b1": {
        "builder_name": "efficientnet_b1",
        "weights_enum_name": "EfficientNet_B1_Weights",
        "family": "efficientnet",
    },
    "efficientnet_b2": {
        "builder_name": "efficientnet_b2",
        "weights_enum_name": "EfficientNet_B2_Weights",
        "family": "efficientnet",
    },
    "efficientnet_v2_s": {
        "builder_name": "efficientnet_v2_s",
        "weights_enum_name": "EfficientNet_V2_S_Weights",
        "family": "efficientnet",
    },

    # ----------------------------
    # Swin Transformer family
    # ----------------------------
    "swin_t": {
        "builder_name": "swin_t",
        "weights_enum_name": "Swin_T_Weights",
        "family": "swin",
    },
    "swin_s": {
        "builder_name": "swin_s",
        "weights_enum_name": "Swin_S_Weights",
        "family": "swin",
    },
    "swin_b": {
        "builder_name": "swin_b",
        "weights_enum_name": "Swin_B_Weights",
        "family": "swin",
    },

    # ----------------------------
    # ConvNeXt family
    # ----------------------------
    "convnext_tiny": {
        "builder_name": "convnext_tiny",
        "weights_enum_name": "ConvNeXt_Tiny_Weights",
        "family": "convnext",
    },
    "convnext_small": {
        "builder_name": "convnext_small",
        "weights_enum_name": "ConvNeXt_Small_Weights",
        "family": "convnext",
    },
    "convnext_base": {
        "builder_name": "convnext_base",
        "weights_enum_name": "ConvNeXt_Base_Weights",
        "family": "convnext",
    },
}


def list_supported_backbones():
    """Return supported backbones grouped by architecture family."""
    grouped = {}
    for name, spec in SUPPORTED_BACKBONES.items():
        grouped.setdefault(spec["family"], []).append(name)
    return grouped


def print_supported_backbones():
    """Pretty-print the supported backbone names grouped by family."""
    print("Supported backbones:")
    for family, names in list_supported_backbones().items():
        print(f"  {family}: {', '.join(names)}")


def _get_torchvision_builder(builder_name):
    """Resolve a torchvision model builder by name with a clear error if unavailable."""
    if not hasattr(models, builder_name):
        raise ValueError(
            f"torchvision.models does not provide '{builder_name}'. "
            "Update torchvision or choose another supported backbone."
        )
    return getattr(models, builder_name)


def _get_torchvision_weights(weights_enum_name):
    """Return DEFAULT weights for a torchvision model if the enum exists."""
    weights_enum = getattr(models, weights_enum_name, None)
    if weights_enum is None:
        return None
    return getattr(weights_enum, "DEFAULT", None)


def create_image_backbone(backbone_name="resnet34", pretrained=True):
    """Create a torchvision image backbone and replace its classifier with a feature extractor."""
    if backbone_name not in SUPPORTED_BACKBONES:
        valid = ", ".join(SUPPORTED_BACKBONES.keys())
        raise ValueError(
            f"Unsupported backbone '{backbone_name}'. "
            f"Choose one of: {valid}"
        )

    spec = SUPPORTED_BACKBONES[backbone_name]
    builder = _get_torchvision_builder(spec["builder_name"])
    weights = _get_torchvision_weights(spec["weights_enum_name"]) if pretrained else None
    backbone = builder(weights=weights)

    if spec["family"] == "resnet":
        image_feature_dim = backbone.fc.in_features
        backbone.fc = nn.Identity()

    elif spec["family"] == "densenet":
        image_feature_dim = backbone.classifier.in_features
        backbone.classifier = nn.Identity()

    elif spec["family"] == "efficientnet":
        image_feature_dim = backbone.classifier[-1].in_features
        backbone.classifier = nn.Identity()

    elif spec["family"] == "swin":
        image_feature_dim = backbone.head.in_features
        backbone.head = nn.Identity()

    elif spec["family"] == "convnext":
        image_feature_dim = backbone.classifier[-1].in_features

        # ConvNeXt applies avgpool before classifier, but the classifier itself
        # performs LayerNorm2d + Flatten + Linear. If we replace it with Identity,
        # the backbone would output [B, C, 1, 1]. Flatten keeps the output [B, C].
        backbone.classifier = nn.Sequential(nn.Flatten(1))

    else:
        raise ValueError(f"Unsupported backbone family: {spec['family']}")

    return backbone, image_feature_dim


class MetadataImageClassifier(nn.Module):
    def __init__(
        self,
        backbone_name="resnet34",
        n_levels=5,
        n_sides=3,
        n_series=3,
        n_classes=N_CLASSES,
        pretrained=True,
        level_emb_dim=8,
        side_emb_dim=4,
        series_emb_dim=4,
        hidden_dim=HIDDEN_DIM,
        dropout=DROPOUT,
    ):
        super().__init__()

        self.backbone_name = backbone_name
        self.backbone, image_feature_dim = create_image_backbone(
            backbone_name=backbone_name,
            pretrained=pretrained,
        )

        self.level_emb = nn.Embedding(n_levels, level_emb_dim)
        self.side_emb = nn.Embedding(n_sides, side_emb_dim)
        self.series_emb = nn.Embedding(n_series, series_emb_dim)

        metadata_dim = level_emb_dim + side_emb_dim + series_emb_dim
        input_dim = image_feature_dim + metadata_dim

        # LayerNorm is used instead of BatchNorm1d because the final training batch
        # can occasionally contain one sample. BatchNorm1d crashes with batch size 1.
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout / 2),

            nn.Linear(hidden_dim // 2, n_classes),
        )

    def forward(self, images, metadata):
        image_features = self.backbone(images)

        # Safety fallback in case a future torchvision backbone returns spatial features.
        if image_features.ndim > 2:
            image_features = torch.flatten(image_features, start_dim=1)

        level_features = self.level_emb(metadata["level"])
        side_features = self.side_emb(metadata["side"])
        series_features = self.series_emb(metadata["series"])

        metadata_features = torch.cat(
            [level_features, side_features, series_features],
            dim=1,
        )

        features = torch.cat(
            [image_features, metadata_features],
            dim=1,
        )

        logits = self.classifier(features)
        return logits


# Backward-compatible alias for existing code that may still reference the old name.
class ResNet34MetadataClassifier(MetadataImageClassifier):
    def __init__(self, *args, **kwargs):
        kwargs.setdefault("backbone_name", "resnet34")
        super().__init__(*args, **kwargs)


def build_metadata_classifier(backbone_name=None):
    """Build the selected image + metadata classifier."""
    if backbone_name is None:
        backbone_name = SELECTED_BACKBONE

    return MetadataImageClassifier(
        backbone_name=backbone_name,
        n_levels=len(LEVEL_TO_ID),
        n_sides=len(SIDE_TO_ID),
        n_series=len(SERIES_TO_ID),
        n_classes=N_CLASSES,
        pretrained=PRETRAINED,
        hidden_dim=HIDDEN_DIM,
        dropout=DROPOUT,
    )


def make_optimizer(model, lr=LR, backbone_lr_multiplier=BACKBONE_LR_MULTIPLIER, weight_decay=WEIGHT_DECAY):
    """Use a lower learning rate for the pretrained image backbone and a higher one for the new metadata/head layers."""
    head_params = (
        list(model.level_emb.parameters())
        + list(model.side_emb.parameters())
        + list(model.series_emb.parameters())
        + list(model.classifier.parameters())
    )

    return torch.optim.AdamW(
        [
            {"params": model.backbone.parameters(), "lr": lr * backbone_lr_multiplier},
            {"params": head_params, "lr": lr},
        ],
        weight_decay=weight_decay,
    )


## Losses and dataloaders

In [ ]:
def move_metadata_to_device(metadata, device):
    return {
        key: value.to(device, non_blocking=True)
        for key, value in metadata.items()
    }


def make_class_weighted_loss(
    train_df,
    device,
    moderate_weight_multiplier=MODERATE_WEIGHT_MULTIPLIER,
    severe_weight_multiplier=SEVERE_WEIGHT_MULTIPLIER,
    label_smoothing=LABEL_SMOOTHING,
):
    """
    Weighted cross-entropy for three-class medical classification.

    Base class weights are computed from the training split. The moderate and severe
    classes receive mild extra multipliers because under-calling pathology is more
    costly than over-calling it, especially for severe cases.
    """
    counts = np.bincount(train_df["target"].values, minlength=N_CLASSES).astype(np.float32)
    total = counts.sum()

    weights = np.zeros(N_CLASSES, dtype=np.float32)
    for cls in range(N_CLASSES):
        if counts[cls] > 0:
            weights[cls] = total / (N_CLASSES * counts[cls])
        else:
            weights[cls] = 0.0

    # Medical-priority adjustment:
    # class 0 = Normal/Mild, class 1 = Moderate, class 2 = Severe.
    if N_CLASSES > 1:
        weights[1] *= moderate_weight_multiplier
    if N_CLASSES > 2:
        weights[2] *= severe_weight_multiplier

    print("Class counts:", dict(zip(TARGET_NAMES, counts.astype(int))))
    print("Class weights:", dict(zip(TARGET_NAMES, weights.round(4))))

    weights = torch.tensor(weights, dtype=torch.float32).to(device)
    return nn.CrossEntropyLoss(weight=weights, label_smoothing=label_smoothing)


def make_weighted_sampler(train_df):
    """Optionally oversample minority-class examples during training batches."""
    targets = train_df["target"].values
    class_counts = np.bincount(targets, minlength=N_CLASSES).astype(np.float32)

    class_weights = np.zeros(N_CLASSES, dtype=np.float32)
    for cls in range(N_CLASSES):
        if class_counts[cls] > 0:
            class_weights[cls] = 1.0 / class_counts[cls]

    sample_weights = class_weights[targets]

    return WeightedRandomSampler(
        weights=torch.DoubleTensor(sample_weights),
        num_samples=len(sample_weights),
        replacement=True,
    )


def make_loaders(
    train_part,
    val_part,
    batch_size=BATCH_SIZE,
    use_weighted_sampler=USE_WEIGHTED_SAMPLER,
):
    train_dataset = LumbarMetadataClassificationDataset(
        train_part,
        transform=train_transform,
        augmentation_transform=train_augmentation,
        augment=USE_DATA_AUGMENTATION,
        use_roi_crop=ROI_CROP_ENABLED,
        use_2_5d_input=USE_2_5D_INPUT,
    )

    val_dataset = LumbarMetadataClassificationDataset(
        val_part,
        transform=val_transform,
        augment=False,
        use_roi_crop=ROI_CROP_ENABLED,
        use_2_5d_input=USE_2_5D_INPUT,
    )

    if use_weighted_sampler:
        train_sampler = make_weighted_sampler(train_part)
        shuffle_train = False
    else:
        train_sampler = None
        shuffle_train = True

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=shuffle_train,
        sampler=train_sampler,
        drop_last=True,  # avoids one-sample final batches and improves training stability
        **dataloader_worker_kwargs(NUM_WORKERS),
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        **dataloader_worker_kwargs(NUM_WORKERS),
    )

    return train_loader, val_loader


def make_test_loader(test_part, batch_size=BATCH_SIZE):
    test_dataset = LumbarMetadataClassificationDataset(
        test_part,
        transform=val_transform,
        augment=False,
        use_roi_crop=ROI_CROP_ENABLED,
        use_2_5d_input=USE_2_5D_INPUT,
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        **dataloader_worker_kwargs(NUM_WORKERS),
    )

    return test_loader


## Training and validation loops

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    n_seen = 0
    all_preds = []
    all_labels = []

    for images, metadata, labels in tqdm(loader, leave=False):
        images = images.to(device, non_blocking=True)
        metadata = move_metadata_to_device(metadata, device)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        logits = model(images, metadata)
        loss = criterion(logits, labels)

        loss.backward()
        if GRAD_CLIP_NORM is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_NORM)
        optimizer.step()

        batch_size = images.size(0)
        running_loss += loss.item() * batch_size
        n_seen += batch_size

        preds = logits.argmax(dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    epoch_loss = running_loss / max(n_seen, 1)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_bal_acc = balanced_accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)

    return epoch_loss, epoch_acc, epoch_bal_acc, epoch_f1


@torch.no_grad()
def evaluate_one_epoch(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    n_seen = 0
    all_preds = []
    all_labels = []
    all_probs = []

    for images, metadata, labels in tqdm(loader, leave=False):
        images = images.to(device, non_blocking=True)
        metadata = move_metadata_to_device(metadata, device)
        labels = labels.to(device, non_blocking=True)

        logits = model(images, metadata)
        loss = criterion(logits, labels)

        probs = torch.softmax(logits, dim=1)
        preds = probs.argmax(dim=1)

        batch_size = images.size(0)
        running_loss += loss.item() * batch_size
        n_seen += batch_size

        all_probs.extend(probs.detach().cpu().numpy())
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    all_probs = np.asarray(all_probs)
    all_preds = np.asarray(all_preds)
    all_labels = np.asarray(all_labels)

    epoch_loss = running_loss / max(n_seen, 1)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_bal_acc = balanced_accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)

    severe_precision = precision_score(
        all_labels,
        all_preds,
        labels=[2],
        average="macro",
        zero_division=0,
    )

    severe_recall = recall_score(
        all_labels,
        all_preds,
        labels=[2],
        average="macro",
        zero_division=0,
    )

    severe_f1 = f1_score(
        all_labels,
        all_preds,
        labels=[2],
        average="macro",
        zero_division=0,
    )

    # Binary medical view: any pathological severity = Moderate or Severe.
    y_true_pathological = (all_labels >= 1).astype(int)
    y_pred_pathological = (all_preds >= 1).astype(int)

    pathological_precision = precision_score(
        y_true_pathological,
        y_pred_pathological,
        zero_division=0,
    )

    pathological_recall = recall_score(
        y_true_pathological,
        y_pred_pathological,
        zero_division=0,
    )

    pathological_f1 = f1_score(
        y_true_pathological,
        y_pred_pathological,
        zero_division=0,
    )

    try:
        epoch_logloss = log_loss(all_labels, all_probs, labels=list(range(N_CLASSES)))
    except ValueError:
        epoch_logloss = np.nan

    return {
        "loss": epoch_loss,
        "accuracy": epoch_acc,
        "balanced_accuracy": epoch_bal_acc,
        "macro_f1": epoch_f1,
        "severe_precision": severe_precision,
        "severe_recall": severe_recall,
        "severe_f1": severe_f1,
        "pathological_precision": pathological_precision,
        "pathological_recall": pathological_recall,
        "pathological_f1": pathological_f1,
        "log_loss": epoch_logloss,
        "y_true": all_labels,
        "y_pred": all_preds,
        "y_prob": all_probs,
    }


In [ ]:
def fit_model(
    model_name,
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    scheduler,
    device,
    epochs=EPOCHS,
    min_epochs=MIN_EPOCHS,
    patience=PATIENCE,
    min_delta=MIN_DELTA,
):
    best_model_wts = copy.deepcopy(model.state_dict())
    best_checkpoint_score = -np.inf
    best_early_stop_score = -np.inf
    best_epoch = 0
    epochs_without_early_stop_improvement = 0

    history = {
        "train_loss": [],
        "train_accuracy": [],
        "train_balanced_accuracy": [],
        "train_macro_f1": [],
        "val_loss": [],
        "val_accuracy": [],
        "val_balanced_accuracy": [],
        "val_macro_f1": [],
        "val_severe_precision": [],
        "val_severe_recall": [],
        "val_severe_f1": [],
        "val_pathological_precision": [],
        "val_pathological_recall": [],
        "val_pathological_f1": [],
        "val_log_loss": [],
        "val_checkpoint_score": [],
        "val_early_stop_score": [],
    }

    for epoch in range(epochs):
        print(f"\nEpoch {epoch + 1}/{epochs}")

        train_loss, train_acc, train_bal_acc, train_f1 = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            device,
        )

        val_metrics = evaluate_one_epoch(
            model,
            val_loader,
            criterion,
            device,
        )

        if scheduler is not None:
            scheduler.step(val_metrics["loss"])

        # Checkpointing is medically oriented: it prioritizes severe recall,
        # but still rewards severe F1, macro F1 and balanced accuracy.
        checkpoint_score = (
            0.35 * val_metrics["severe_recall"]
            + 0.25 * val_metrics["severe_f1"]
            + 0.20 * val_metrics["macro_f1"]
            + 0.20 * val_metrics["balanced_accuracy"]
        )

        # Early stopping uses a more general validation-progress signal.
        # This avoids stopping only because severe recall temporarily decreased
        # while F1/balanced accuracy continue to improve.
        early_stop_score = (
            0.50 * val_metrics["macro_f1"]
            + 0.50 * val_metrics["balanced_accuracy"]
        )

        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_acc)
        history["train_balanced_accuracy"].append(train_bal_acc)
        history["train_macro_f1"].append(train_f1)

        history["val_loss"].append(val_metrics["loss"])
        history["val_accuracy"].append(val_metrics["accuracy"])
        history["val_balanced_accuracy"].append(val_metrics["balanced_accuracy"])
        history["val_macro_f1"].append(val_metrics["macro_f1"])
        history["val_severe_precision"].append(val_metrics["severe_precision"])
        history["val_severe_recall"].append(val_metrics["severe_recall"])
        history["val_severe_f1"].append(val_metrics["severe_f1"])
        history["val_pathological_precision"].append(val_metrics["pathological_precision"])
        history["val_pathological_recall"].append(val_metrics["pathological_recall"])
        history["val_pathological_f1"].append(val_metrics["pathological_f1"])
        history["val_log_loss"].append(val_metrics["log_loss"])
        history["val_checkpoint_score"].append(checkpoint_score)
        history["val_early_stop_score"].append(early_stop_score)

        print(
            f"Train loss: {train_loss:.4f} | "
            f"Train acc: {train_acc:.4f} | "
            f"Train bal acc: {train_bal_acc:.4f} | "
            f"Train macro F1: {train_f1:.4f}"
        )

        print(
            f"Val loss: {val_metrics['loss']:.4f} | "
            f"Val acc: {val_metrics['accuracy']:.4f} | "
            f"Val bal acc: {val_metrics['balanced_accuracy']:.4f} | "
            f"Val macro F1: {val_metrics['macro_f1']:.4f} | "
            f"Val severe precision: {val_metrics['severe_precision']:.4f} | "
            f"Val severe recall: {val_metrics['severe_recall']:.4f} | "
            f"Val severe F1: {val_metrics['severe_f1']:.4f} | "
            f"Checkpoint score: {checkpoint_score:.4f} | "
            f"Early-stop score: {early_stop_score:.4f}"
        )

        checkpoint_improved = checkpoint_score > best_checkpoint_score + min_delta
        early_stop_improved = early_stop_score > best_early_stop_score + min_delta

        if checkpoint_improved:
            best_checkpoint_score = checkpoint_score
            best_epoch = epoch + 1
            best_model_wts = copy.deepcopy(model.state_dict())

            save_path = OUTPUT_DIR / f"{model_name}_best.pt"
            torch.save(best_model_wts, save_path)
            print(f"Saved best model: {save_path}")

        if early_stop_improved:
            best_early_stop_score = early_stop_score
            epochs_without_early_stop_improvement = 0
        else:
            epochs_without_early_stop_improvement += 1
            print(
                f"No early-stopping improvement for "
                f"{epochs_without_early_stop_improvement}/{patience} epochs."
            )

        if (epoch + 1) >= min_epochs and epochs_without_early_stop_improvement >= patience:
            print(
                f"Early stopping at epoch {epoch + 1}. "
                f"Best checkpoint epoch was {best_epoch} with checkpoint score {best_checkpoint_score:.4f}."
            )
            break

    model.load_state_dict(best_model_wts)

    history_df = pd.DataFrame(history)
    history_df.insert(0, "epoch", range(1, len(history_df) + 1))
    history_df.to_csv(OUTPUT_DIR / f"{model_name}_history.csv", index=False)

    return model, history_df


## Plotting and test-report helpers

In [ ]:
# ============================================================
# EVALUATION HELPERS
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_curve,
    auc,
    roc_auc_score,
)
from sklearn.preprocessing import label_binarize


def plot_history(history_df, title="Training history"):
    epochs = history_df["epoch"]

    plt.figure(figsize=(20, 4))

    plt.subplot(1, 4, 1)
    plt.plot(epochs, history_df["train_loss"], label="train")
    plt.plot(epochs, history_df["val_loss"], label="val")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Loss")
    plt.legend()

    plt.subplot(1, 4, 2)
    plt.plot(epochs, history_df["train_balanced_accuracy"], label="train")
    plt.plot(epochs, history_df["val_balanced_accuracy"], label="val")
    plt.xlabel("Epoch")
    plt.ylabel("Balanced accuracy")
    plt.title("Balanced accuracy")
    plt.legend()

    plt.subplot(1, 4, 3)
    plt.plot(epochs, history_df["train_macro_f1"], label="train")
    plt.plot(epochs, history_df["val_macro_f1"], label="val")
    plt.xlabel("Epoch")
    plt.ylabel("Macro F1")
    plt.title("Macro F1")
    plt.legend()

    plt.subplot(1, 4, 4)
    plt.plot(epochs, history_df["val_severe_recall"], label="severe recall")
    plt.plot(epochs, history_df["val_severe_precision"], label="severe precision")
    plt.plot(epochs, history_df["val_pathological_recall"], label="moderate/severe recall")
    plt.plot(epochs, history_df["val_checkpoint_score"], label="checkpoint score")
    plt.plot(epochs, history_df["val_early_stop_score"], label="early-stop score")
    plt.xlabel("Epoch")
    plt.ylabel("Score")
    plt.title("Medical-priority metrics")
    plt.legend()

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


def load_best_model(model_name, backbone_name=None):
    """Load the saved best checkpoint using the selected backbone architecture."""
    model = build_metadata_classifier(backbone_name=backbone_name).to(DEVICE)

    model_path = OUTPUT_DIR / f"{model_name}_best.pt"
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    return model


def compute_ovr_auc_and_accuracy(y_true, y_prob, class_names):
    """
    Computes class-wise one-vs-rest AUC and one-vs-rest accuracy.

    acc_ovr for a class means:
    class vs all other classes accuracy = (TP + TN) / all samples
    """
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    n_classes = len(class_names)
    labels = list(range(n_classes))
    y_true_bin = label_binarize(y_true, classes=labels)

    auc_by_class = {}
    acc_ovr_by_class = {}

    for class_idx, class_name in enumerate(class_names):
        y_true_class = y_true_bin[:, class_idx]
        y_prob_class = y_prob[:, class_idx]

        # One-vs-rest prediction from predicted class probabilities
        y_pred_class = (np.argmax(y_prob, axis=1) == class_idx).astype(int)

        acc_ovr_by_class[class_name] = (y_true_class == y_pred_class).mean()

        if len(np.unique(y_true_class)) < 2:
            auc_by_class[class_name] = np.nan
        else:
            fpr, tpr, _ = roc_curve(y_true_class, y_prob_class)
            auc_by_class[class_name] = auc(fpr, tpr)

    try:
        macro_auc = roc_auc_score(
            y_true_bin,
            y_prob,
            average="macro",
            multi_class="ovr",
        )
    except ValueError:
        macro_auc = np.nan

    try:
        weighted_auc = roc_auc_score(
            y_true_bin,
            y_prob,
            average="weighted",
            multi_class="ovr",
        )
    except ValueError:
        weighted_auc = np.nan

    return auc_by_class, acc_ovr_by_class, macro_auc, weighted_auc


def make_report_table(y_true, y_pred, y_prob, class_names=TARGET_NAMES):
    labels = list(range(len(class_names)))

    report_dict = classification_report(
        y_true,
        y_pred,
        labels=labels,
        target_names=class_names,
        zero_division=0,
        output_dict=True,
    )

    report_table = pd.DataFrame(report_dict).T.reset_index()
    report_table = report_table.rename(columns={"index": "class"})

    auc_by_class, acc_ovr_by_class, macro_auc, weighted_auc = compute_ovr_auc_and_accuracy(
        y_true,
        y_prob,
        class_names,
    )

    report_table["acc_ovr"] = np.nan
    report_table["auc_ovr"] = np.nan

    for class_name in class_names:
        report_table.loc[report_table["class"] == class_name, "acc_ovr"] = acc_ovr_by_class[class_name]
        report_table.loc[report_table["class"] == class_name, "auc_ovr"] = auc_by_class[class_name]

    return report_table, macro_auc, weighted_auc


def plot_confusion_and_roc_side_by_side(
    y_true,
    y_pred,
    y_prob,
    class_names=TARGET_NAMES,
    title_prefix="Test",
    save_path=None,
):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    y_prob = np.asarray(y_prob)

    n_classes = len(class_names)
    labels = list(range(n_classes))

    cm = confusion_matrix(y_true, y_pred, labels=labels)
    y_true_bin = label_binarize(y_true, classes=labels)

    auc_by_class, _, macro_auc, weighted_auc = compute_ovr_auc_and_accuracy(
        y_true,
        y_prob,
        class_names,
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # ----------------------------
    # Confusion matrix
    # ----------------------------
    ax = axes[0]
    im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
    ax.set_title(f"{title_prefix}\nConfusion matrix")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    tick_marks = np.arange(n_classes)
    ax.set_xticks(tick_marks)
    ax.set_xticklabels(class_names, rotation=45, ha="right")
    ax.set_yticks(tick_marks)
    ax.set_yticklabels(class_names)

    thresh = cm.max() / 2 if cm.max() > 0 else 0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(
                j,
                i,
                format(cm[i, j], "d"),
                ha="center",
                va="center",
                color="white" if cm[i, j] > thresh else "black",
            )

    ax.set_ylabel("True label")
    ax.set_xlabel("Predicted label")

    # ----------------------------
    # ROC curves
    # ----------------------------
    ax = axes[1]

    for class_idx, class_name in enumerate(class_names):
        y_true_class = y_true_bin[:, class_idx]
        y_prob_class = y_prob[:, class_idx]

        if len(np.unique(y_true_class)) < 2:
            continue

        fpr, tpr, _ = roc_curve(y_true_class, y_prob_class)

        ax.plot(
            fpr,
            tpr,
            linewidth=2,
            label=f"{class_name} AUC={auc_by_class[class_name]:.3f}",
        )

    ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1, label="Chance")
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_title(
        f"{title_prefix}\nROC / AUC | Macro={macro_auc:.3f}, Weighted={weighted_auc:.3f}"
    )
    ax.legend(loc="lower right")
    ax.grid(alpha=0.3)

    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=200, bbox_inches="tight")

    plt.show()

    return macro_auc, weighted_auc